# Robot data quality: controlled Lift policy experiment

五组 × 三个训练随机种子 × 100 次相同初始状态评测。先通过独立 pilot 再冻结步数。BC 使用本项目 PyTorch 实现，数据来自 robomimic，环境为 robosuite 1.5.1。此实验不代表真实 DROID 机器人性能。

运行前在“代码执行程序 → 更改运行时类型”中选择 T4 GPU 和运行时版本 **2026.07**，以固定 Python 3.12。安装器拒绝在 Python 3.13 上编译旧版 NumPy。

首次安装如果更换了已载入的 NumPy，安装单元会明确要求“重启会话，然后全部运行”。依赖保留在当前云端虚拟机中，重启后顺序运行即可；不要在混合 ABI 的旧内核里继续执行。

训练与安装仅在 Colab；断点保存到本账号云端硬盘项目目录。免费 GPU 不可用时可先检查 CPU 耗时，配额不足就停止并保留状态，不购买算力。

源码包由 `scripts/build_notebooks.py` 从仓库中的可读模块生成，SHA256 为 `186e11c2824e1f9066bb09c704a82a46387d7a408bd12bdaeb47db9e94c86154`。Base64 仅用于在 GitHub 发布前让笔记本自包含，不包含数据、密钥或个人文件。

In [ ]:
import os, sys, subprocess, base64, io, zipfile, hashlib
from pathlib import Path
print('Cloud Python:',sys.version,flush=True)
assert (3,10) <= sys.version_info[:2] <= (3,12), 'Select Colab runtime version 2026.07 (Python 3.12) before installing these pinned dependencies.'
loaded_numpy_version=getattr(sys.modules.get('numpy'),'__version__',None)
os.environ['MUJOCO_GL']='egl'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ['PYTHONDONTWRITEBYTECODE']='1'
os.environ['PIP_CACHE_DIR']='/content/robot_audit_cache/pip'
os.environ['NUMBA_CACHE_DIR']='/content/robot_audit_cache/numba'
for key in ['PIP_CACHE_DIR','NUMBA_CACHE_DIR']: Path(os.environ[key]).mkdir(parents=True,exist_ok=True)
print('Installing pinned simulation dependencies...',flush=True)
subprocess.run([sys.executable,'-m','pip','install','--only-binary=numpy,scipy,h5py,scikit-learn,mujoco','numpy==1.26.4','robosuite==1.5.1','mujoco==3.3.7','h5py==3.14.0','scikit-learn==1.6.1','requests==2.32.5','PyYAML==6.0.2','imageio==2.37.0','imageio-ffmpeg==0.6.0'],check=True)
from importlib.metadata import version
if loaded_numpy_version and loaded_numpy_version != version('numpy'):
    raise RuntimeError('Dependencies installed. Restart the Colab session, then run all cells to load the pinned NumPy ABI consistently.')
source_root=Path('/content/robot_audit_source');source_root.mkdir(exist_ok=True)
bundle=base64.b64decode('UEsDBBQAAAAIAAAAIVyK95XqUQAAAFYAAAAbAAAAc3JjL3JvYm90X2F1ZGl0L19faW5pdF9fLnB5FcrBCYAwDADAv1OEDFDqAC7hAiE2eRS00SYV3F59HhwirrZZgOhhzaNzVGtwDd5rPMBDagA3AR+lqDucXaWWPyVEnIhu7f6JCBbAnOaUcXoBUEsDBBQAAAAIAAAAIVwe2MV0UQYAAMwOAAAYAAAAc3JjL3JvYm90X2F1ZGl0L3BhdGhzLnB5nVdbd9o4EH7nV6jZB9spMb099JBD9yRAGvY0wKHQy2n3+Ci2AAXbciU5JJvNf98ZyTcobXfXD4kuM6PRNzPfiKOjo7M4JpkUNyzUZCu5ZopQyYgUQrOIsLss5iHX8f0p+cjTSGxVKaUF6XetLENtFvlHR0etpRQJCYJlrnPJgoDwJBNSE5qmQlPNRaparWLtRom0HAtVjtQ61zwuZ5ol2ZLHzNrNqF7H/Lo0OoVpq9U/618Og+H4A+mRhxaBz5kPr6ZOlzio7bRhvjudjqaB1RqMZriRcbO++LC7nN86bWvw8iK4nFwNcXGdr1Y8XS1pyFAHdxbnVm9vu7POrxsGPg3nh8XumC4sDc7mZ++H8/eH5SKqqWJaNWy+G84m55N55VzMpLgWxtx8MutfVhtayHCNy58Gb4tLllt30aq0eDV915+ML0ZvCwASqrNYaIAcVceLq/OzXYTSPLmmuDeafp5fTsbFMs/u9Vqkpdk/FrA7nAXWdql7k4MUk51QpEu+QiOl3Gwxno+uhvuCMk81T1hp1Z64eD+cnQNoJozm1JNcMWnCbASmn43H09nwYvTJSoU0XKOZx1arFbElCWkqUh7S2MUE65q88sjJGzPomtMkUyK+hYromST0i7nrmV3N7jTsKC3dUtBu/FZVTULvwQgURUr0mkFhaZZGLDqJWbrSa6IyFscQaLIUklCScKVwFq55HJHtGivA2gNXQV8SMJrGgkYklIxiPXKtwDXJUu2TsZAJjflfrLZLlwBhcQ2YF9Zu8jQ0RXkKXnEFVsESnACCMb0zniITdCLJbxl4w8KNb1T5EkrWT2nCSK8HeaAd8CwyQPhKU6nVluu163yF7/evXx3PbKPlH4ksxn2Us3BbyA1aGAMXlb686v7ptRo7JdQQxTCmSpGPQm5UBqVijWBog4CnXAeBq1i8bJvLdDFO5G9jGP6NRQp3MP/a5FsOLBWsrrtkCeBiTF8+a7iERny0ARt10hgPzSrEDlBh6S2XIvVXTLuOrc+zxWA0D2aTyRwSE+X9cBu5HnyV7R8gWh3pmxj4eZYx6XrkCcgMuk7tmwGGcsXIBxrnbCilkK5Tpt8himeAA0fiRi4fdH3noDNPGs5g/DDJK6e8ZiRdB2sZ8lp3sPw6OsnsYC0SZkfbMkJ2mqCo5/3qEguY07RqRiSMRR51+iNSmWsb1yhR9wpInkRcwmWFvG/eyThtA3x9jyD0CE+1W4acHJPnz168Oj5+6bWq9MFaL1MHKkJDBJrps0cS+KHGTnbUOdOpbOwAjZ4bTuEqKAUCLRog/wKfpWNSmamQZqwOtc31h9Lk42lVML0HPO+JwiUQ6j1UR8Ha4SxopqRx9n9l46xkUZOGFitgHZFrxSO2l4NFlWemyZcBgf6XZwYagz1WbX0ocid6a4IFwSXV08CH4xLl7iEJqQs1isSN9zexNrXs+LZDeDZk3gElP9lAkrmWcFVvLnM4ld1xpQOxMdNdrZoWvqCHfxbdwhqrRRtiRfsaTMbzj7PRfHgOrbE/GQwd1HWeOz/RWcwvXn8nVr6kfByA74UH9dWLW3fMK6lBTIhqgafr4BukI+kWy9eMIdtCphSLcAWogMNTBZ8oxJEMH2mqA0XGZEpjXIvFSjl7Uag9QKz/DbBVNuSKrlidDXDSd+1D5YmbIUtpF8kqUNgV8UoZXqhmV7mKxbXrHEPQIe0zLEZECzi6Ps70v8AQRkEKNIo49k8ad/F0wPSZ8SQCXq1dgQdJVGaZdbmBrmTYfuyr14+42gT1rQqORZlmURp7TxuHkzffsduhUpzZB1TJGi+fkbf8vOILo9w1xnsP+PcR6+hbDm0CV+rT9iii4cZT8qKmUXAKHf+lI84oVflyCcwOEbd4GE4/JRvGMrCITq4ZjQCM5ABBPDiIktPd60yQbXgJi4Zj74UZyBN4D5SL+6i1d5zFz0GHKnmcgBFTKAHmLC4+bLr7bX/jmRzb7JDQ42OdSvjb5yeNpU1ukTYPcFzRX3aKZq+nGH4unoL/gaagXZevW2zngQnKnXkS+3YM8XV8jezQVPINmQf4RHPxWn6UJ5lyzQXaAEAER/dewLEQZvhJSFXIee+Cxgp2AQqax7oHN4diimOxDVKa2l0PVUIRwWu15+R6efJ671wgmBjyxHjotf4BUEsDBBQAAAAIAAAAIVw2BnaGWwYAAKEQAAAbAAAAc3JjL3JvYm90X2F1ZGl0L2Rvd25sb2FkLnB5lVdbb9s2FH7Pr+D8MFGZIztGWnQa1KFDC2zAtgZtgT4EgUBLRzEX3UZSiTPD/33nkJQsO3YvepFEHp7rdy6cTCbXsq4hZ223LGV2oZtOZcDy5rEuG5HrKVOgu0osS2CtUEaKkhWyBM1EnbOsqQ3Uhq2EXoGOJpPJWaGaiqVp0ZlOQZoyWbWNMkhdN0YY2dT67Myv0alSLvvff3RT999GVuBYoYisUwqlRI6n7ll+WikQ+XXTlO/WkHWmUVMmdJo1VVuCgdydb4UhIf2ha/x1G50qcT1CozT0u/92jYFBPwX/dqANKmwPRMRqkP65Ufe6FRmSn+VQML0SixcvOdHEVkrILl4zbVR8xvBZsaS3N/Kkod14lGZllYyaFmoeqGUQohlIXOcluMP0FI1i2aqr75msmTSgeCmqZS5iTxmRM/gVO2eX88XV+fkinLJlEIQ7DlaNqGtzYYBbVk4DBejWGrdWsM7lHVqMqjmjjALgCtomJksICw9SYwz9b9GUOSj7Y60tpTY3uczMrZOKLkazi2BlTKvj2WzV3d3J+q5Ar0VZMxOtnKEyQoPRsw2J2c5IIn07OdvZxsnY/qowxErLB0iM6uDHUlbSJJfz+TxwRjSPGmXd3DqnrhCiJH5nPQKnRfABEvVxje7AcCSaWrg1nUn45Ysp+3kehs+ORUpIDSlGIdWI407zEQ3KjmCNmZDztQ3UmoI0HCVg85DJgq1vAvPUQnDLkoQFlEfBjo3z1nCqlPW9UzGokXkwZZtt6P6RMtiLHWngQ1aAyVYpseaPOt7BlIJ3Ko6YXOopZhQ5G0f6iD1/0snu3wSE0sA5OEf/4QbaTYuc8M4DiuVMiUcE8MxKs6/SaQrrFjLMSjwla8MtS2eNlv8BWtd7vSdMMU162Y6wLPTYC43MvRe0zCETCqlJr4hSKtVdUcg1twvum/3EgugBlCwk5DYo/jgGxpLBGvGLgbWlzfPcX3TcMP48xFdKmlMke5V3aKvAkPIkJLJ1lPf8KE1TgwFFF2RNjumQBJ0pLl4FI9ShRsTB2dkHC92aJEPorD47IuvE8LgyI6DQgSGAVPuorlb3uVTc/ejkE6YXQoLMTpt7++sU68v/N3iZSL1zv1IDDvMf4d+UD/slwFZlLEOYqFoUkExmk3Dr0p6STRgDVWtsyon6DviLUdEjYO85oikKlIcqeXMOw4mu73eG0EOJdWO+xwbJPKfXJzxuvU5Fg33oaiow75RqFC+C61EbxbMZQK536UFKxAzNL7ej2rAv8odTUaZnhQADRcVwE3wgfwQxun/5ZEAnG8dgexFsR+yseZvtM062NT0vll5A4t9TqiEgKg+coZQu5lN2ucCspm7WV7Xn6jpwfr3IHvcEJcFw2p3BASAHctFi/vK4uD0YzI+S+K7scOAas1gGhz4LHk8068Nnr3kP+lIXT/0IxRejzn2aET2+3z8qud/HTz2otD9ioCwRzl9C7OFzFMFv/XTo0Qv5N8L3RMp9Ec5Ogfcfe9l/1P2ANwypx+W5UQYjPExmVvgzpfb6jS3y7uBIK9r6xuT++PsbFMYqqSuBjdir9gtrMeqgHkZDNEJC1poEYJWLDpTvPYVFscRKaQvsPoXvMJuA6iamOL2mbNcs4qFT4Krt3LSC6etaRTwYRyvWQ0HsTZ8eRUXQtS7R04F87J/98tF3OwtS1+5sL8y7qtWcdJ+i9TkCP6FB9Vkr/GLvooeQhzWfD+Xpg/t4Z9et1R4zB9mEEe8bBjbLqxNh3Vul+Ea6BGh5JWu+OD/3HKbslW/aR7Cwy5JCYKWnWWU8oVWifvq+CU0CUu9G7Cl7xJNYgGMaqBAKV0dncJHnkvxhGzfe5Hgl1nw+tUMYjqMWDLd4kvGvD3PrfgYMD1L4eBlBV38Xz4N+i51jmKW9/f7KpKNsBdl9SoOB4DsL+5lYd6UZXQaolj+/LJIjUu/CxL9tMW+RaHTv8pdOTDXawClniTcPvhuzMQ566tNvl3J2bA3j/dHZ2mNXRjbtEod2nTjaHl9ludciPBzqskbRRO22I2c5P8weWoxE29L9xB05KDaKwIBl/f3nv/98/+Yt25TY8Ny5EOcv+usDsKWCRixuJmTT5Ha0QEiY3M4u4WUcXRZb9tdvOK4XZadXoynSJ7PGOzTkXsiU3cNT4u6zbB2PMHH2P1BLAwQUAAAACAAAACFclkxYOakZAADjUAAAGQAAAHNyYy9yb2JvdF9hdWRpdC9wb2xpY3kucHm1XHlz3LaS/1+fAuvUW5LOiDoSbZJxJlV6sr3rej5UtpOqV1NTLIrEaBjxCkFKGqvmu28fAAlySFnJZlU+eACNRqOPXzdAPXv27KLI66pIUxkLlWRNGtZJkQt5X8oqyWRez0Qs0+RKVmEt061QsgzxUqyrIhMvP35481JEaZhkyj84+FVJJariqsiSLInE/7x8fQa9syJXdUV0lbhL6o2oNzKpqKFqEqB14p/5J0Lmt0lV5DioiIp8nVz7B+8LcfHry/OZuNx+LqpoMxNFZfiEq0SJJCuLqgbum7xOUhECN0UTCxgwyZP8+kjehimQy7Iwj/2DZ8+eHRDnQbBu6qaSQaApiDDPi5q5PDjQz6Ki3JrrTag2IAhz+7sqcnNdKHNVg8x4gDKssbmhfgm3LdnNWUc2b7JyK0Il8pI7qptUhlXuy1zJ7CqVhsIbVfDivC4qqep+47KSZVVEUimYs+nxsbhqVP0pClNZHXB7Py7u8rQIY9NmLetoE6yTVM6E2oSnZ/81A9lJeXDw4Z+fgn+9+vcnsRCuU1z9LqPamQkHV60+DqRcB2WhBk/+aEK70XWVlKBHwR/Y0ju4+PD+5ZvPbz68Z5oRsJ5ja1jsOsySHNQqpt6wVEUWxFVR4m3YxEnd3hVVCB351js4OIjlWpRFmkRbHD1N6m2wliEurXLDCOU1B9H6eRxWVbj1xOEv1u38QMAPKHgdAkvwPE7Wa91tJsL7RC2OPWrzu6xurCbUpd8iC6/zpG5iyc1SmE967edFlfVan3DrSgKLOTYkRtwlXGUgDxefXBnWPW+GTVQdmwd0n4X3e+2I6t6PodoyxwRAUmgtsns+E/5PZx11q/3jhDUbKB7N7MgsxC9A/ccpUs97M2yFugLPU29LuViDvtZmqS1d0R3UXMRJVC/Bx8yspYXu64pbLPwTUG4p48X3px4veQVWQsvEuuYD6bBJ6wCeu9iSF+lGbhU0U+RgzHD8KirA38A7FBZQT/LareBR7KYyd7GfJ563DHged1IyBSsCX7VABvxoUySRpNYzJjgDvSjTMJKL12GqpOfXoNmqds2YnZ0AiYebuQh9dFGuJ9bgDW9AdsCJ0Iz64Foz5Xo76lxXDXjehViu6BbbJzOcIfaQ4IXIw7uGRS0n/Cmq5BqVGTprykvotmrfJ2uRiH+IU7FYiON5b42THJZGqv58UUKGJqiMlmDv6dER6mJfFj3CtiiIm6UeatUfa3l44s8gvoA2qOSLXNDguqk3a6fmg+sr5fJk5fWHuYGmQNA5PQ7Aj0UQmQIWQaDK5EYqp20tgcX54yxqhYNY2050Js76TmQ48Dq5lYGqZRlESRVB1KsMA1miwNtdU7js2KBV9kPwuqCKD44sE1XE0pnjQoPzRLJ4A/9NGCM6WOatnSlFBeilA6DPD9o5gI5ebWvwtp7nb+R9nFxL1NhJ+lFRVU0JQvnaAHsCnBhp1/Oodq8Zy0O7DlbtQAcLaAdhBbXZpUCkuVGPuo0R4rBKtk+aoMWExrxKr0Pft6Cu0hj8+BvxeSMhWNUwC7DeSkYStEMZoxRFnm59apPkGK0RxzGLAJMA2wDCEWF13RDAAgKh0IHSJ/L3rJ+qDqMbdzkVUvuLstKOB30ITmvltaRs8OF6/hpCOGCyXK0xIN7rkAm6iW5lAG3cPIB/kwwhnlp8d3w8s0SMC9NJVUMFYLqWCxQxhIDg9+JKLU5pTDOSipAyDHVIY/p0H6gwK1OYlG7UAQ3yrjCfZbJiT4kTpFh9jcvmMrkZGdLCgdEBpzne8pBWbr5ij2vBGFxuWbvT8UZ8K068PxMQmGMLDCHL1bK1eOa7Qr5JB3ZGf7XSY3ONwDh+DhRxCMvmfd0fMe4ebJtTdLo1YekW+bAJmNiEweOGlBMaWAR2oyNYSPAvDdD1H6dvY8u/NIBFYLfT2JLNNYB1RLyNYmfzBAeoF0uRdGm1Hcpb7BctnphyqCB4WUMnoIdYu11iPQPsf+sNp6G1oIUI08RZZtppo7uGVCNKFA+AxC2r+U9bAN4Rz+nplIFwmKb/V7Jai6bo2lb55whjo5JM4aGviQYfts+82dAadBProTcb6ptuY7OklagLbXrddOyZDdXL5EMVZunSvSM3gpm0jl8lRyt4sHTYwzuM4yAkVFsMAPIefCZpyz37DindcukgeQdAFF3e8vLDrXMLqftRmqzro3LjkC3cA2XIdcEBARyE5phaOiuTIuXJGlw7DNTlncTl9BDEGXdHutD1Tvl45WJK7TpxWIdHVXgHox91ZPjaHhtigqxuQy0+EzPBHHcz+EtNqDaBubn/GvnCQXAhgTKk5xtYutSCeDgudOfHS2LD6TCxzG+DTFILLBT4mHIrFxv5YV1XChw1tMBwYvjDHzb+Pj64h1gRgS/QGJ9WxQyahermiDo5q+XcInQLUTv+s4So04BQsqbA1THmgdHgk24Erw97oaWS4rcwbeSrqioq17kkResXgoQqAVeAQYY34TWEzpYE+SlOSixpfGvNqD+cvawWygYlj8BLEyRbotg5SXCgOczwBifZIRdT6Vi1KbofKkw9MWBT9vndaR+e23kQDGgNoN+QGKeojNhxGx0mQSple2ZYi/lOSjuyHcvvGZiEZmSsHsGGUTljSE5Y1cka2qoj7RQ8P7uJk8pFL5LXavG5aiSYIginDoobujWY9CXaAWXcgEQrSZEQy21JjUFSvMnFBYC7K6z5gV5UMgN2iW/AqVyquyuqG1UCxmEcqgtTZRLdpLKzy0lmj7S3i/3yJgXOC8h9XOfuatxsmawfN1kJ+ZGtPeCB+z6iXctW8AyK9IowJDILOR41nG51oHl3g56zVWh4093AG+M74Lm5hKXlibQLh15lRBYt8g84E8RmjtayQV9WOdMzYCnS3E2vh+fPuREwRf5t3nrw4Xx5OtR7CF4sT4EhrxOxmTwBWmODezDHscrClkSAFIDwElNbR9eDAUW/hUj0QqhtDtpVJ5FYJzkgnMNb9EaxXkXRWYhydO5YVljAcS4/vH1z8W9x+fHV5fnHc6xUisvzT59AFOu0URut9xxjA3TpELu0+/t7tLaa0FoNALTy0shGHzQ3uazRjtwkLxsAIUlmYhxe93mssZBu4wp64Oe5/0n+0YCYkzB122dvAVmGlU0XcnNYybbBR/n21+l0v/0ZEqRK899IypqtRfFzmG8wWdJSysIbGYAKuUb6nJej/wWtiIPbJJaFTrV6Mmv3KjiDv8PITVgKwEIsZUlVOEN02Znwiq+5g/G735Ce6l2PuJCcncv7sgAP+a75vYiKV/mto4g3jIawsjH6zKRW6HMgiDYI+nyLGb8pYwx2m1ABFsxjWcmKJ4KuQwXFeq0iAHN599ae85jMGyWDKMS6YIAJdUsM+kDyK2vzpCXoZMS7juZdsovTsKqqWutakfq4Ko8ILwceMIQ9f85TbVeTuAiQMNCHlrrIMreqJWWSQ2KkRU27EgJ4EpRSUx2ESwCVwtmK9012uQW5X6fFVZiKj+//2398KjCoj9XGKwD+1zLHEiosC1UjHi8vj/WwhUOEcXqumW276RWorLgZw/ZagDBw65u49lGHBC2GwB+ukrRgAaqAWmkEi4U4qpZYxWKgLkOTpl9L99TCfcAtjjpmXUi1w05RUyGg6CiTOVfbPqbD4Uj128GIjKYGYPC7AebEH1BR5GFCJ/aa33IRbR8sAp2l3u46pGWBZIJSfEhnzDYYVXtxxB4Sa2Hk/mihgqhX60G52Iz03JOfZh4pJHCRKI5WLrcCDJqmLobMT+12pxU8qRaB+E5kicpCSKacvVG1tE05eFBexXKW4s0n1DbQL/8aRYdzxoIdDAkyQQ54G8b5ef2j403UYPenTDRrWRL6bWAq+KBz0jPhH5+J51qtrP5rrCmnA43AvgAWlXS7hqykZnJ6rrqQx+LmFstjSkX1zQlmrZ9IPQSKG2KdoCkrSFCQkgDZAhxAjS9SwKdABvBJUW2dR+GThVLYQFsQheXBukGg42gw0QNBIAt4BYrQm7CD29ZsmQYhTVgCaoieCK9dv55O8x/QZpnLOECIEuB+pqysrQ0z37lAvNNHSJ/evPv17fnnDx/Fx1efXn0W5+9fivMLQkqf3n3416tpvMTg9yridL91EC2ENpEYdUZNARaO62jheZMF9aaSmFbrtIpfAkJttOgsRz3pwcG0uV/UxCHYYBDehkmK9VzXcjJWE4t+gBbakeJGV2F0A2qksDUAkCuZR5sspM1jipdWUwywWMCpMlhABXgVCF4XFSxKptwux4rlbRJhPOFefOs6yI3zKP/sppyobJyxnYcu4lpJzWrZ3ljF/KGT7Lr2MqhVzx9aGwLbR2lMDD9F7BvxvqgyyCq+sCekgw3tJqU5+NFL5mhvBLQyuac0oCoUJKtpaldCdfQlU4jkX51z18jK/Fam/CVDVPOa9n/NSLxfrvcAZ9Zz2hLnx+D7ktL1j0+6FeFFB2AH7MF6uu69OKQBvCPoaHw2t9LFh5nWpAX/163MkNb26d3NDk6bf9y3W6kzse22VSFsuHa/AvKvLPkiq3Z4euKfx2Hm8hYNZqMZWofCGJRWC//YCKAOcfd+7Ty067cjg3zAf3bsqKONjG7KIiG0YdKw9X4e9gDEdn5ZO32wdGy8Q0fHp/qHsv1CiZXKolHtLAh4dV1wd7sENBbxvhVLYCbuZHK9qQGUg1YOd7d58khHu3PanDEDLR16bxcNW1E+0qltY3c0U+2aoe91ehv73OYX9spfK/RddCJPlLiSW1gdqvNUmFcqTNDZAlsbvWpiwBxW+a9z8ACVNRbp+KO3+MZZ+eDVbNgBvLJ0fdRcAmvsIAnuawoIcNhxEhHPbIy+L/LB5r3lU6eYacn0eCGBEQjHo2B+VgD9Ik8iDVwAw6iAyhHQAodt8TVKeATwdsGQZtmep2AGMaZhXMY6yz1YiXty+uPMG7VUra6oGOg0F6xo7n17cKJrhky2Q0AMWzc5dQpTP4OQha/djhTY+QiNTivhbxFcVyFG3TqoiyCHaVslRDMiRU5I8GJ3jApByd5yu/jo2xNP/EOcHB8f0+kToQX57QnejihtT/7k2FwaG8JwGG0A9g7USi8ppXMP2vbm2kYtS6PKVmtj8x7fvUZkYXPN42TRsFXzuV6E654O0uas0b65rauDdtNWQcgA9W+Kh4x3hjlkAdsxcR0T2NSVIHhie3unA/fwpvP9UyPYG5UWCuQK9VyjQR1GcY0QB7erN0U0BFEkuCXWVFQafPf2EotMh/BXYHVpBrEj35h67gvx7tOrFwKDjgkwM3GFydQCDMnZ9V2CRDQa0jaZFRQIQ6tmDZ7Ndfw6Kx1vxJOo8JaMugbv3xLyxun7esvfiiP9lozF187nj+dv3gsKYKRRiwfWq90RXagdCWzx0Ilt7p+td7D4YalkvHgYuqhD7b3m/sl6p4wXeeD/dwNYj5yYjcmWUQ32MXtpUAUHYN8KjASgtZvTm98bAHFfIEx+jyc+vlaR65ICY59PicAEhqcD8B6cIdpLS+dXM6GfWeq+snrvRWHd3MTtF7oZishtq0hT5RNLSLZANIhqamCrK6YwLLJ0E7Qg0ymEegoMopNm6oE5foY3z1a7gFLRh46VHae2mnfERAmdprQ3OmE8H5MzgJT3WB5gT6Q6DMUeSDP+DdimajJp9AYDFC4NZTe4T9QBCwqTVO+QYL5UwNIoAlhB/Ip9k1xvR+BxdKCc1hrfW9wuq+6ETPsYxqsYJBiIwiIBqEDxhFaSURJzfrUl4GkO35ik3VnNxQh9cxLTqm5gm7abFf+the+uIeEnUxmUv5J1n4RmaqRGRgpjSia62bLti8qJZaUkb+Rwk3W8ttZ27TsoU1ihJOovFJnGiAVYzRCL4SFB/XaiLNWP402EB+R7ibj5IavqlyXN6vSBmfZRIwXI/9eK4tc2pPGHP60gJ5jkOosMMtrx3+cWf8I+FNTZHmTTAX2U4OpKJCSWxochKFhhhqkNAbeSPZ9b7zME05mJgP/AKHZFkBJac4jvEKLuyUgJEa3RcnoE5emYMKmxJ34Wp/SMVmj8AHJvfY3iG03kjQt3Q8FgQbtId0lcb/hSb33g7sPCCa9lXt8m8s7xlvP54clIjRetEAgH5KwCrWxTou90EYPpaJMr8J83QzuneeyT1DExyYDPpPBvT3ETUd+NNOZWWZIRLPlqVHhozXznZ+X3DlgrbxqJdakWJ4Ojy7xkfL7QgnhaZ7pHK4PuyMtq0KcVi1xo7z154XnfBU8eMe788LxzUQgtWerw9KooUlffGlyuOmA+WVK1nVEfHvZ9KwuhLxhTOtbBEU+83kpIiHABqqKo8Tw6a/Zw1W2lx2xnQtENLnz12/lbDQsfrK4ACyl67Iz2ScCGEHbdCmI9P3lmHRs1vUZg316Nflif17iQSZhDaRirn3Yk7cX0ZpYuTtPBrTZ3poNhwPJjB1pwTw9kw6dcUR/OnmtkUy6f7W2EAe7RKMfahC/uVB/m4GA9mNMJ5L7kI55EUx//5QhSju670dmdkecQ8tsX7cGKVb/wQccagTlPH7kbtiYn+QAtegiFV7q4o9gGvXfYu893X8UYZtGZOaI1MM59gmNqrIl44j8W4mRfhfeLSu/GKkaKaq0E/jrIOMANGhUuTBe/LMpBkKo4ceA6gPl0aUmTM65ib1bDT0S0Qi55lBX5PeNN+FnneAIcEJ5XlAc6g4MytITwuMTtq1g3E78shP+DOQaKIJnE3VYorQrk9FkTS+m7Smey7upi+5XNvSxrnwF0NlbOZYg9OeMybtxr7VgvGUh6WS61FNky9PXz097dd6u5bQe6+882n/vf5PTQba9KPLFF1X6oeDbTQ3g9iaTY9wkZ71cNv7NYeqihpl2yHVfY+6G60tlOZs1W18mjXzi6vV/5BO3ds1xbm0dn5GAVOKnQt/Tp+T+MK/3IGJ38oOXZzIr0WpzzCSnvrH3jv89gScKPmuxXxW52W434l4Y9OlpqnYbVm3b6tWcphQn7l2/efvhsYvviAVmZ+9+td3pWiwf+fyygawPquO+bzfR5Q8gWvkj8JqCoi6hIrZODpYWqWrHylFtp0bHSURD3WCFvxgU7q1onsOyNtTyH6ocpH4HEboRUeNfhkNxWZ5DOxMgtQg0AwVE1l7bfu4+Xvd0Q8CLYYQ9N4esjfoqetbsi6D3EPy90vFqHSSrjFyIuaOuhavCrEfSlsgK3UetzsdAFo11UZOBIEkQc7cfO4NoyePZl5AiQOYQI8R7wP/wp40mMpSMKwtTBSZ8noylHIygsvx/vwyWm7VNZJnYfg01tQCKJcD+L0IhQ3xfIagZryyzZlSNd8dFhbo27iLjLFft4VPk13mp4bn9uhzKgpv51VTTl1da1c5W9VAUycvyetIW77NnQl7jRTCg+Yh+hFK2P3qmaQMjFbA/zAQ/9vZg+a8/sQAofy3uCSS116K7f4uNSozkLFfphvrWD+JjkXrPYLHklOapZKsFdmLEEfndqfy12L/CwhRQMZ+9xx8Wcj1G+Y46IsBPoY+RpVDLqO7x9PK3lwqvTB50knwdDoQUGuyeJgNhI8OAPmyIkeTBztD/mbGKb8jFo3zLSHd+x4vwjbxHm91/vgf2eGLrCG0pgCrU/bf15HnwMiA9TWDIwXHWgsW8lVkbh+WNJs583efJHg9U+/NJ7CPzHWHwzOJfFnOljG51SQqwPyS46L2kqv7bt+3URROq2U8RBGG7VGBphaYMMz94UKJPbom7dA9253Gg58Ao2IOGPLtImy9Wi50loIwaeGcjW1vb07zkgqTzy6wnGvuS4KooaP60pLVdu1HPobHTF/QKlicIDKICHLOZ0IKa1ic4YFPpPUlOFwoH2MdqEXo7WRoi6323IKF39pdqvBi8D3fWNZnhdChBYJ0whqljOjE/qxv0irX4YtENa3/7zM/7NAnxt/U6BPvbpnTi1KOCcNAG8HPbvoyTNiykBkaKAE4yWVFvnqx67FobrFrDtHusqsqs7eYz2vW7Fh1GtIzLYH3rY+1bC7LDMhPkYdf/rCGrwglwj/54fHWpLWVHszSO5h6Gsw44mqPDvg2Eb08CZ9IBxMyUEPb9hnZQcEt8rB+55fm2B7NkCk0HZDdusao86YTDUzUzWmwLxqeZP1HfF4V24NbbSrdZ8oP9cjt5Q5ap/uvSFQHVGbMLfnO9PDj/e1ZnNnwQi7Ei0evCx8YAclIvnrngXEmYt83rhaOTj7H9n0/uy6SHCr1wHWRCnnaTXy2hl1PErn2s4UfLTGXRecm8UHXZufwONf3x6hodYp17/9AO8Xu3GANXeV8v6s9wMUnwV2B9rm9/eEV7LgMC/aqcDSOa5ttXuA95DfmCT8KbmOz0ozjwoy27yNBZO0R6K7vsjDYXzuIRHf/7qgCzuR+eoP2L+ayK1v4A2EtVf306kyWYDV6eWdomcM18Cm/hBn9KVbo7hoO2n3n6q2//U6/Wb314dtholzi8uXl1+Pn9/8chBZs0WfsdoZWB0LOBa6l8YQidrOqhC9yZHW3NL2v3ThSKnd6Jw+Ok2o86015GOmFvdHvk6ZKw7paL2qMPa/OiYJvG0xx1LRsc6Q55rddPg8m9OFLR4MWBxl2W/4LDqx+sx6HkJLU2inlTKPpo4uSVBEzLeybUOMNPxUXPHO7xWFWHkmxcmQHCB4T6Gke6IIpMwU+uBupFtxqeUNjFXnbUUdQwd+Zbj68VNQ2Ms8em9s0uC5sVIpbMnOpABzZ+iqxbK/pnNcV38X1BLAwQUAAAACAAAACFcCUV18nQBAACVAgAAFAAAAGNvbmZpZ3MvcHJvamVjdC55YW1sdVJbjtwgEPz3KTjAagwYbDNXiSLU5jFDhAcC2Nnd0wc8o6wTKZI/6K5qylR1NkZfEaNdTMa6d2kTrCZf0YC7n1soIG/LUexOmyC/WnOXYY3eXDuE8qaUyXWIUIxxbVhwfkumDdY6hy1V/MysR4SSieGKfjzC7Qa9TsFp+cJf8O6yC496rQU14nmaBWegRzsaPHK6CEUIcDyO7VOU4mPMw2J8nTn9xv/UXvg/aowOxHChQCzCwiyWeRqU1YxozgVlUN80zIOCsxruYvBOfTSpp1AKS1jd6lT/5yQ1FMim5O4sNzELmMxsJJZNQmlhNeFLEyIjwQOunoqR4qG9x1W/0U4uvPfOlj7eex9+Se1WuRN+uWvLK0uFlLZY6uUtTFUOFXxphpQE7iFzjbyG9Y1Mb4iKN8SG7xUzO/gnJHOBVKqF+JnmgZjoctBtM8jRjM6H8jdf4BPyxT9yycXE/NyHVi5Q1F1m92na0sy1cw/JfR72V8JvUEsBAhQDFAAAAAgAAAAhXIr3lepRAAAAVgAAABsAAAAAAAAAAAAAAKSBAAAAAHNyYy9yb2JvdF9hdWRpdC9fX2luaXRfXy5weVBLAQIUAxQAAAAIAAAAIVwe2MV0UQYAAMwOAAAYAAAAAAAAAAAAAACkgYoAAABzcmMvcm9ib3RfYXVkaXQvcGF0aHMucHlQSwECFAMUAAAACAAAACFcNgZ2hlsGAAChEAAAGwAAAAAAAAAAAAAApIERBwAAc3JjL3JvYm90X2F1ZGl0L2Rvd25sb2FkLnB5UEsBAhQDFAAAAAgAAAAhXJZMWDmpGQAA41AAABkAAAAAAAAAAAAAAKSBpQ0AAHNyYy9yb2JvdF9hdWRpdC9wb2xpY3kucHlQSwECFAMUAAAACAAAACFcCUV18nQBAACVAgAAFAAAAAAAAAAAAAAApIGFJwAAY29uZmlncy9wcm9qZWN0LnlhbWxQSwUGAAAAAAUABQBhAQAAKykAAAAA')
assert hashlib.sha256(bundle).hexdigest()=='186e11c2824e1f9066bb09c704a82a46387d7a408bd12bdaeb47db9e94c86154'
zipfile.ZipFile(io.BytesIO(bundle)).extractall(source_root)
sys.path.insert(0,str(source_root/'src'))
print('Source bundle verified; importing simulator...',flush=True)
import torch, robosuite, mujoco, yaml, importlib
import robot_audit.policy as policy_module
importlib.reload(policy_module)
print({'python':sys.version,'torch':torch.__version__,'robosuite':robosuite.__version__,'mujoco':mujoco.__version__,'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})


In [ ]:
from google.colab import drive
drive.mount('/content/drive',timeout_ms=300000)
from robot_audit.paths import Workspace
cloud_root=Path('/content/drive/MyDrive/robot-data-quality-success-prediction')
cloud_root.mkdir(exist_ok=True)
ws=Workspace(cloud_root);ws.setup()
config=yaml.safe_load((source_root/'configs/project.yaml').read_text())
from robot_audit.policy import policy_main
ws.json('reports/cloud_source_manifest.json',{'bundle_sha256':hashlib.sha256(bundle).hexdigest(),
    'source_sha256':{str(p.relative_to(source_root)):hashlib.sha256(p.read_bytes()).hexdigest() for p in source_root.rglob('*.py')},
    'python':sys.version,'torch':torch.__version__,'device':'cuda' if torch.cuda.is_available() else 'cpu'})
ws.path('reports/cloud_requirements.txt').write_text(subprocess.run([sys.executable,'-m','pip','freeze'],capture_output=True,text=True,check=True).stdout)
print('CLOUD STORAGE PREFLIGHT PASS',cloud_root)


In [ ]:
policy_main(ws, config, 'prepare')
policy_main(ws, config, 'smoke')
policy_main(ws, config, 'pilot')

In [ ]:
policy_main(ws, config, 'run')

In [ ]:
policy_main(ws, config, 'summarize')
print('COLAB SEQUENTIAL RUN ACCEPTANCE PASS')
print(ws.path('reports/policy_results.json').read_text())